# 📗 Langfuse 프로덕션 트레이싱: 배포된 서빙 API가 요청마다 남기는 기록으로 비용과 실패 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 교안을 배우는 이유</strong></p>
<p style="margin:6px 0;">응답이 느려졌다는 전체 수치만으로는 누구의 어떤 질문이 느렸는지 찾기 어렵습니다. 이 교안에서는 배포된 API가 요청 한 건마다 남긴 기록으로 입력·출력·사용자·배포 버전·토큰을 확인하고, 느림과 실패의 원인을 좁힙니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/19_why_tracing.png" width="1050" alt="전체 응답이 느려진 것을 확인한 뒤 요청별 기록에서 사용자, 질문, 배포 버전, 토큰을 찾아 원인을 좁히는 이유를 보여 줍니다.">

#### **프로덕션 trace(production trace)**

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배포된 서버가 실제 사용자 요청마다 남기는 기록</strong>입니다. 정상 처리된 요청에는 사용자, 창구, 배포 버전, 질문과 답, 걸린 시간, 토큰 사용량이 남습니다. 실패한 호출은 답이나 사용량 없이 오류 상태만 남을 수 있습니다. 예를 들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-app</code> 키로 노무 창구에 &quot;일주일에 40시간이 넘게 일하라고 하는데, 이렇게 근무해도 되나요?&quot;를 보내면, 정상 처리된 경우 질문과 답, 배포한 커밋의 sha, 걸린 시간(초), 출력 토큰 수를 확인할 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 메트릭은 전체 요청의 합과 분포만 보여 줍니다. 예를 들어 메트릭으로는 &quot;p95가 느려졌다&quot;까지만 알 수 있고, 느린 요청이 어느 사용자, 어느 배포 버전의 것이었는지, 그 요청에 어떤 질문이 들어왔는지는 요청마다 남긴 기록에서만 찾을 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 느린 요청의 원인 찾기, 사용자별 토큰과 비용 합계, 배포 버전 사이의 비교입니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">span:</strong> 시작과 끝이 있는 요청 처리 단계의 기록입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">generation:</strong> span 안에서 모델을 한 번 호출한 기록입니다.</li>
</ul>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/20_production_trace_concept.png" width="1050" alt="프로덕션 trace 그림: 사용자 요청 하나가 서빙 API의 span과 그 안의 모델 호출 기록(generation)으로 남고, 사용자, 창구, 배포 버전, 토큰, 지연이 함께 붙음">

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day79-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span>

1. 서빙 API에 Langfuse trace를 붙이고, 사용자, 창구, 배포 버전을 기록하도록 배포합니다.
2. 요청 한 건의 trace에서 입력, 출력, 토큰, 지연을 읽고, 실패한 요청이 오류 레벨로 남는 것을 확인합니다.
3. 서빙 벤치마킹 단원의 비용 식으로 출력 토큰 단가를 계산해 Langfuse에 모델 단가로 등록합니다.
4. Langfuse API로 이번 부하의 기록을 읽어 정확한 p95와 사용자별 토큰, 비용 합계를 계산합니다.

</div>

## 1. 실습 준비

### 1-1. Langfuse 프로젝트와 키

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01의 VM과 GPU Pod를 그대로 씁니다. Langfuse에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-prod</code> 프로젝트를 만들고 발급한 키와 주소를 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에 저장합니다. 같은 이름의 줄이 있으면 새 값으로 바꿉니다. <a style="color:#245fa0;text-decoration:underline;" href="../Langfuse_%ED%82%A4_%EB%B0%9C%EA%B8%89_%EA%B0%80%EC%9D%B4%EB%93%9C.md">Langfuse API 키 발급 가이드</a>를 참고하세요.</p>
</div>

```dotenv
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

### 1-2. 노트북 준비

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01과 같은 준비 코드(경로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>, 주소와 헤더, 부하 함수, 실행 기록 함수)에 Langfuse 클라이언트와 기록 읽기 함수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">fetch_generations</code>)를 더해 한 셀에 모았습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">fetch_generations</code>는 5-2에서 설명합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 앞 교안에서 준비한 서버 연결을 이어 쓰고, Langfuse 기록을 읽을 설정을 더해 이후 측정에 재사용합니다.</p>
</div>

In [ ]:
# ---- 라이브러리, 경로, 설정값(교안 01과 같습니다) ----
import json
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

day_dir = Path("..").resolve()       # data, images, output 이 있는 일차 폴더(day79_서빙운영_관측_SLO)
data_dir = day_dir / "data"
output_dir = day_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

SERVED_NAME = "qwen3.5-4b"            # vLLM의 --served-model-name

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# ---- JSONL 읽기와 노무 질문 ----
def load_jsonl(path):
    """path가 가리키는 UTF-8 JSONL 파일을 읽어 딕셔너리 목록으로 돌려줍니다.

    파일의 각 줄을 JSON 딕셔너리 하나로 바꾸고, 원래 줄 순서를 유지해 리스트에 담습니다.
    질문 파일은 보통 한 딕셔너리에 question과 기대하는 메타데이터를 담습니다.
    """
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")

# ---- 주소와 헤더 ----
# GPU 파드의 vLLM 주소(RunPod proxy). 2-2 에서 VM 의 .env 에 씁니다
VLLM_URL = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/v1"

# VM 의 서빙 API 주소와 관리자 키 헤더
VM_URL = "http://" + os.environ["VM_HOST"] + ":8080"
ADMIN_HEADERS = {"X-Admin-Key": os.environ["ADMIN_API_KEY"]}

# 노트북이 VM 의 서빙 API 응답을 기다리는 최대 시간(초). VM 의 vLLM 시간 제한 90초보다 길게 둡니다
REQUEST_TIMEOUT = 120

# ---- GitHub 실행 기록 ----
# GitHub API 는 로그인 없이 부르면 IP 하나에 시간당 60번이라 같은 와이파이의 교실 전체가 나눠 씁니다. 읽기 전용 토큰을 붙이면 학생마다 시간당 5000번입니다
GITHUB_HEADERS = {"Authorization": "Bearer " + os.environ["GITHUB_READ_TOKEN"], "Accept": "application/vnd.github+json"}  # 배포 단원 교안 00 5-2 의 읽기 전용 토큰(값은 출력하지 않습니다)

# 이 저장소의 GitHub REST API 주소(실행 목록은 최대 60초 늦게 바뀔 수 있습니다)
GITHUB_API = "https://api.github.com/repos/" + os.environ["GITHUB_OWNER"] + "/serving-deploy"


def recent_runs():
    """GitHub Actions의 최근 워크플로 실행 20개를 읽어 표로 돌려줍니다.

    ci, cd, rollback 실행을 함께 읽습니다. 각 행은 실행 한 건이며, 이름, 제목, 커밋 sha,
    시작 이벤트, 상태, 결론, 실행 링크를 담습니다. GitHub가 돌려준 최신 실행이 표의 위쪽에 옵니다.
    """
    response = httpx.get(GITHUB_API + "/actions/runs", params={"per_page": 20}, headers=GITHUB_HEADERS, timeout=30)
    response.raise_for_status()
    runs = pd.DataFrame(response.json()["workflow_runs"])
    return runs[["id", "name", "display_title", "head_sha", "event", "status", "conclusion", "html_url"]]


# ---- 부하 보내기 ----
import asyncio
import time

# 동시에 보낼 요청 수(서빙 벤치마킹 단원의 기본 부하 16)
MAX_CONCURRENCY = 16
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def run_limited(task):
    """동시에 실행할 요청 수를 제한하고, 한 요청이 끝나면 그 결과를 돌려줍니다.

    task에는 실행할 비동기 요청 함수가 들어옵니다. semaphore 자리가 날 때까지 기다린 뒤
    task를 실행하고, 요청 함수가 돌려준 결과를 반환합니다. 요청 오류는 여기서 감추지 않고 그대로 전달합니다.
    """
    async with semaphore:
        return await task


async def send_chat(client, path, body):
    """서비스 키가 설정된 비동기 client로 질문을 보내고 요청 결과 한 행을 돌려줍니다.

    path는 /v1/labor/chat 같은 상담 주소이고, body는 question을 담은 딕셔너리입니다.
    반환 딕셔너리에는 HTTP 상태 코드와 내 PC에서 잰 걸린 시간이 들어갑니다.
    상태 코드가 200이면 답의 끝난 이유, 출력 토큰 수, 답 글자 수도 넣습니다.
    200이 아니면 서버가 돌려준 오류 코드를 넣어, 429·502·504 응답도 다음 질문을 계속 보낼 수 있습니다.
    네트워크 연결 자체가 끊기거나 응답을 JSON으로 읽지 못한 오류는 그대로 전달됩니다.
    """
    started = time.perf_counter()
    response = await client.post(path, json=body)
    seconds = time.perf_counter() - started

    reply = response.json()
    row = {"상태 코드": response.status_code, "걸린 시간(초)": round(seconds, 2)}
    if response.status_code == 200:
        row["끝난 이유"] = reply["finish_reason"]
        row["출력 토큰"] = reply["usage"]["completion_tokens"]
        # 생각이 생성 상한을 다 쓰면 답(answer)이 None 으로 온다. 그때 답 글자 수는 0
        row["답 글자 수"] = 0
        if reply["answer"] is not None:
            row["답 글자 수"] = len(reply["answer"])
    else:
        row["오류"] = reply["error"]["code"]
    return row


# ---- Langfuse 클라이언트와 기록 읽기 ----
from langfuse import get_client

# .env 의 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL 을 읽습니다(에이전트 관측 단원과 같은 방법)
langfuse = get_client()


def fetch_generations(start, end):
    """UTC 시각 start부터 end까지 시작한 모델 호출(generation) 기록을 표로 돌려줍니다.

    Langfuse API를 100건씩 페이지를 넘겨 가며 끝까지 읽습니다. 반환 표에서 모델 호출 하나가 한 행이며,
    trace id, 시작 시각, 사용자 id, 배포 버전, 창구 태그, 오류 레벨, 지연(초), 출력 토큰,
    비용(달러)을 확인할 수 있습니다. 실패해 토큰 사용량이 없는 호출은 출력 토큰이 비어 있습니다.
    """
    rows = []
    cursor = None
    while True:
        # fields: 기본 칸(basic) 에 사용량(usage), 지연(metrics), 태그(trace_context)를 더해 받습니다. 한 번에 100개씩
        page = langfuse.api.observations.get_many(
            type="GENERATION",
            from_start_time=start,
            to_start_time=end,
            fields="core,basic,usage,metrics,trace_context",
            limit=100,
            cursor=cursor,
        )
        for observation in page.data:
            # 실패한 호출은 사용량이 비어 있습니다
            output_tokens = None
            if observation.usage_details is not None:
                output_tokens = observation.usage_details.get("output")
            rows.append({
                "trace_id": observation.trace_id,
                "시작 시각": observation.start_time,
                "user_id": observation.user_id,
                "version": observation.version,
                "tags": observation.tags,
                "level": observation.level,
                "지연(초)": observation.latency,
                "출력 토큰": output_tokens,
                "비용(달러)": observation.total_cost,
            })

        # 다음 쪽이 없으면 cursor 가 None 입니다
        cursor = page.meta.cursor
        if cursor is None:
            break

    # 기록이 아직 하나도 없어도 열 이름은 있게 만든다(빈 테이블에서 열을 고르다 멈추지 않게)
    columns = ["trace_id", "시작 시각", "user_id", "version", "tags", "level", "지연(초)", "출력 토큰", "비용(달러)"]
    return pd.DataFrame(rows, columns=columns)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 필요한 값이 모두 있는지, Langfuse 키로 로그인되는지 확인합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">auth_check()</code>는 키와 주소로 Langfuse에 로그인되면 True를 돌려줍니다.</p>
</div>

In [ ]:
# .env 에 필요한 값이 있는지 이름만 확인합니다(값은 출력하지 않습니다)
env_check = {}
for name in ["VLLM_API_KEY", "ADMIN_API_KEY", "GPU_POD_ID", "GPU_PRICE_PER_HOUR", "VM_HOST", "GITHUB_OWNER", "GITHUB_READ_TOKEN", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    env_check[name] = name in os.environ
env_check["Langfuse 로그인"] = langfuse.auth_check()
display(env_check)

### 1-3. 노무 사용자 키 발급

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">키 값은 교안 01 노트북의 메모리에만 있어 이 노트북에서 가져올 수 없습니다. 이 노트북에서 쓸 키를 새로 발급하고, 교안 01과 같은 pro 사용자 셋에 하나씩 배정합니다. 사용자마다 그 키를 헤더에 담은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.AsyncClient</code>를 만듭니다. 키 원문은 출력하지 않습니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 사용자별 키로 부하를 보내 trace의 사용자 ID를 요청·토큰·비용 합계와 연결합니다.</p>
</div>

In [ ]:
from datetime import datetime, timezone

# 교안 01 의 노무 pro 사용자 셋에 오늘 쓸 키를 발급합니다
labor_users = ["labor-app", "hr-team", "mobile-app"]
issued_keys = {}
issued_rows = []
for user in labor_users:
    issue_response = httpx.post(VM_URL + "/v1/admin/keys", headers=ADMIN_HEADERS, json={"user": user, "plan": "pro"}, timeout=REQUEST_TIMEOUT)
    issued = issue_response.json()
    issued_keys[user] = issued
    # 키 원문(api_key)은 빼고 상태 코드와 key_id, user, plan 만 테이블에 담습니다
    issued_rows.append({"상태 코드": issue_response.status_code, "key_id": issued["key_id"], "user": issued["user"], "plan": issued["plan"]})

# 사용자마다 base_url 과 서비스 키 헤더를 담은 AsyncClient(부하는 이 셋에 나눠 보냅니다)
user_clients = {}
for user, issued in issued_keys.items():
    user_clients[user] = httpx.AsyncClient(base_url=VM_URL, headers={"X-API-Key": issued["api_key"]}, timeout=REQUEST_TIMEOUT)
labor_clients = []
for user in labor_users:
    labor_clients.append(user_clients[user])

display(pd.DataFrame(issued_rows))

## 2. 서버에 trace 붙이기

### 2-1. 바뀐 파일 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">trace, observation, tag는 에이전트 관측 단원에서 배웠습니다. 이번에는 기록을 남기는 쪽이 노트북이 아니라 VM의 서빙 API이므로, 서버 코드에 모델 호출 기록, 요청을 감싸는 span과 속성, Langfuse 키 설정을 더합니다. 바뀐 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">교안_02_Langfuse_프로덕션_트레이싱/serving-deploy/</code>에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델 호출 기록(generation)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">모델 호출 하나의 입력 메시지, 출력, 토큰 사용량, 모델 이름, 걸린 시간을 담은 observation입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">OpenAI 래퍼:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">langfuse.openai</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">openai</code>를 감싼 모듈입니다. import 한 줄만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">from langfuse.openai import AsyncOpenAI</code>로 바꾸면 호출 코드는 그대로 두고 상담 요청마다 generation이 남습니다.</li>
</ul>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/services/backend.py(바뀐 줄)</strong></div>

```python
# services/backend.py
from langfuse import get_client
from langfuse.openai import AsyncOpenAI
langfuse = get_client()
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">langfuse = get_client()</code>:</strong> 서버가 뜰 때 Langfuse 클라이언트를 하나 만듭니다. 키와 주소는 컨테이너 환경 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">LANGFUSE_PUBLIC_KEY</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">LANGFUSE_SECRET_KEY</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">LANGFUSE_BASE_URL</code>에서 읽습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">키가 없으면 꺼짐:</strong> 배포 전 테스트(CI의 image 작업)에는 Langfuse 키가 없습니다. 이때 클라이언트는 경고 한 줄만 남기고 기록 없이 동작하므로 테스트가 그대로 통과합니다. 키 없이 뜬 서버가 돌려주는 응답 헤더의 trace id는 0만 32개인 값이 됩니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">span과 propagate_attributes</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">span:</strong> 시작과 끝이 있는 작업 하나의 기록입니다. 상담 요청 하나를 span <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-chat</code>(전월세는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-chat</code>)으로 감싸면, 그 안에서 부른 모델 호출의 generation이 span 아래에 붙어 한 trace가 됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">propagate_attributes</code>:</strong> with 블록 안에서 만들어지는 observation에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user_id</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">metadata</code>를 함께 붙이는 Langfuse SDK 함수입니다. span과 generation 모두에 같은 값이 붙으므로, span으로 거르든 generation으로 거르든 같은 요청이 나옵니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스트리밍 주소는 그대로:</strong> 이 교안은 일반 상담 주소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/{desk}/chat</code>)에만 span과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">propagate_attributes</code>를 붙였습니다. 스트리밍 주소의 generation에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user_id</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>이 붙지 않으므로, 이 교안의 기록 읽기와 비교는 일반 상담 주소의 요청으로 합니다.</li>
</ul>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/22_trace_attributes.png" width="1050" alt="trace 속성 그림: 상담 요청 하나의 span 안에 generation이 있고, 둘 다에 user_id(key_id), tags(창구), version(이미지 태그), metadata(plan, think), environment가 붙음">

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/routers/metered_router.py(가져오는 줄)</strong></div>

```python
# routers/metered_router.py
from langfuse import propagate_attributes
from ..services.backend import ask_vllm, langfuse, make_system_prompt, open_vllm_stream, to_chat_response
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/routers/metered_router.py(상담 엔드포인트)</strong></div>

```python
# routers/metered_router.py
# 경로 매개변수 desk 에 Literal 을 붙여 노무, 전월세 창구를 엔드포인트 하나로 처리한다
@metered_router.post("/{desk}/chat", response_model=ChatResponse)
async def metered_chat(
    desk: Literal["labor", "lease"],
    request: ChatRequest,
    response: Response,  # 응답 헤더(X-Trace-Id)를 붙일 응답 객체
    client: dict = Depends(check_rate_limit),
    connection=Depends(get_db),  # lifespan 이 연 DB 연결
):
    """상담 질문 하나의 답을 돌려주고, 그 요청의 토큰 사용량을 키별로 기록한다. 요청 하나를 Langfuse trace 하나로 남긴다."""
    # 0) 관측 단원: 상담 요청 하나를 span 으로 감싼다. 그 안에서 부른 vLLM 호출은 langfuse.openai 가 generation 으로 남긴다
    #    propagate_attributes 의 값은 이 span 과 안쪽 generation 에 모두 붙는다. user_id 는 key_id(Langfuse 는 US-ASCII 200자 이하만 받는다)
    with langfuse.start_as_current_observation(as_type="span", name=f"{desk}-chat", input={"question": request.question}) as span:
        with propagate_attributes(
            user_id=str(client["key_id"]),
            tags=[desk],
            version=settings.image_tag,
            metadata={"plan": client["plan"], "think": str(request.think)},
        ):
            # 1) desk 에 맞는 시스템 메시지로 vLLM 에 묻는다(desk 가 labor, lease 가 아니면 422)
            completion = await ask_vllm(SYSTEM_PROMPTS[desk], request.question, request.think, max_tokens_think=THINK_LIMITS[desk])

        # span 의 출력에 끝난 이유를 남기고, trace id 를 응답 헤더로 돌려준다
        span.update(output={"finish_reason": completion.choices[0].finish_reason})
        response.headers["X-Trace-Id"] = span.trace_id

    # 2) 그 요청의 토큰 사용량을 기록한다. client 는 check_rate_limit 을 통과한 키다
    usage = completion.usage
    await record_usage(
        connection, client["key_id"], f"/v1/{desk}/chat",
        usage.prompt_tokens, usage.prompt_tokens_details.cached_tokens, usage.completion_tokens,
    )

    # labels(desk=desk) 는 이 창구의 카운터를 고릅니다. inc(...) 는 이 답의 출력 토큰 수를 더합니다.
    OUTPUT_TOKENS.labels(desk=desk).inc(completion.usage.completion_tokens)
    return to_chat_response(completion)
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">user_id=str(client[&quot;key_id&quot;])</code>:</strong> Langfuse SDK의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user_id</code>는 US-ASCII 200자 이하만 받으므로 한글 사용자 이름 대신 key_id를 문자열로 씁니다. key_id의 주인은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /v1/admin/keys</code>로 찾습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">tags=[desk]</code>:</strong> 창구 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease</code>)입니다. 화면과 API에서 창구별로 거를 때 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">version=settings.image_tag</code>:</strong> 지금 실행 중인 이미지의 태그이고, 값은 배포한 커밋의 sha입니다. 배포 버전마다 기록을 나눠 비교할 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">metadata</code>:</strong> 요금제와 생각 모드 여부입니다. 값은 문자열로 넣습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">응답 헤더 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">X-Trace-Id</code>:</strong> trace id는 trace 하나를 가리키는 고유 문자열입니다. 서버가 이 id를 응답 헤더에 담아 돌려주면, 요청을 보낸 쪽이 그 요청의 trace를 Langfuse에서 바로 찾아 열 수 있습니다(3-1).</li>
</ul>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/core/config.py(더한 줄)</strong></div>

```python
# core/config.py
    # 지금 도는 이미지의 태그(커밋 sha). compose.yaml 의 environment 가 넣고, Langfuse 기록의 version 으로 쓴다(관측 단원 교안 02)
    image_tag: str = "local"
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Settings</code>에 필드 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image_tag</code> 하나를 더했습니다. 값은 배포 단원에서 compose 파일의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">environment</code>에 넣어 둔 환경 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>를 읽기만 합니다. 의존성은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">langfuse</code> 한 줄이 늘었습니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/routers/metered_router.py(version 전달)</strong></div>

```python
# routers/metered_router.py
            version=settings.image_tag,
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">라우터가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">settings.image_tag</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">propagate_attributes</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>으로 넘깁니다. 배포할 때는 커밋 SHA가 요청 span과 그 안의 generation에 함께 기록됩니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · pyproject.toml(더한 줄)</strong></div>

```toml
# pyproject.toml
    "langfuse>=5.2.0",             # 요청마다 trace 남기기(관측 단원 교안 02)
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · deploy/env.example</strong></div>

```text
# deploy/.env 의 모양입니다. 값은 교안 02 노트북이 만들어 scp 로 올립니다. .env 는 저장소에 올리지 않습니다(.gitignore)
# 이미지 이름: ghcr.io/<GitHub 계정 이름(소문자)>/serving-deploy
SERVING_IMAGE=ghcr.io/your-github-id/serving-deploy
# RunPod GPU 파드의 vLLM 주소(끝에 /v1)
VLLM_BASE_URL=https://YOUR-GPU-POD-ID-8000.proxy.runpod.net/v1
VLLM_API_KEY=
ADMIN_API_KEY=
# RunPod proxy 는 100초를 넘기면 끊으므로 그보다 짧게 둔다
VLLM_TIMEOUT_SECONDS=90

# Langfuse 프로젝트 키와 주소(관측 단원 교안 02). 내 PC 의 실습자료/.env 와 같은 값
LANGFUSE_PUBLIC_KEY=
LANGFUSE_SECRET_KEY=
LANGFUSE_BASE_URL=https://cloud.langfuse.com
# 이 서버가 남기는 기록의 환경 이름(Langfuse 화면의 Environment 필터)
LANGFUSE_TRACING_ENVIRONMENT=production
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Langfuse 세 줄:</strong> 내 PC <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>와 같은 프로젝트 키와 주소입니다. 그래서 서버의 기록이 1-1에서 만든 프로젝트로 들어갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">LANGFUSE_TRACING_ENVIRONMENT=production</code>:</strong> Langfuse SDK가 인식하는 환경변수 이름입니다. 서버 컨테이너에 전달되면 SDK가 별도 코드 지정 없이 trace와 observation에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">production</code>을 기록하므로, Environment 필터에서 이 값을 골라 봅니다.</li>
</ul>
</div>

### 2-2. VM .env 아홉 줄로 다시 쓰기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01에서 쓴 다섯 줄 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">env.example</code>의 Langfuse 네 줄을 더해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/vm_serving.env</code>에 씁니다. 6절에서 시간 제한만 바꿔 두 번 더 쓰므로 함수로 만듭니다. 키가 든 파일이므로 변수 이름만 출력해 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">push 전에 올리는 이유:</strong> push가 시작한 배포 작업은 VM에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose up -d</code>로 컨테이너를 새로 만들고, 이때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy/.env</code>를 읽습니다. 새 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>가 먼저 올라가 있어야 새 컨테이너가 Langfuse 키를 가지고 시작합니다.</p>
</div>

In [ ]:
def write_vm_serving_env(timeout_seconds):
    """VM의 서빙 API가 사용할 환경 변수 파일을 만들고, 포함한 변수 이름만 돌려줍니다.

    timeout_seconds에는 VM 안에서 vLLM 응답을 기다릴 제한 시간(초)을 넣습니다.
    함수는 저장소 이름, 서버 주소, 키, 제한 시간, Langfuse 연결 정보를 output/vm_serving.env에 씁니다.
    키 값은 화면에 출력하지 않으며, 반환 목록에도 값 없이 변수 이름만 담깁니다.
    """
    serving_image = "ghcr.io/" + os.environ["GITHUB_OWNER"].lower() + "/serving-deploy"
    vm_env_lines = [
        "SERVING_IMAGE=" + serving_image,
        "VLLM_BASE_URL=" + VLLM_URL,
        "VLLM_API_KEY=" + os.environ["VLLM_API_KEY"],
        "ADMIN_API_KEY=" + os.environ["ADMIN_API_KEY"],
        "VLLM_TIMEOUT_SECONDS=" + str(timeout_seconds),
        # 오늘 더하는 네 줄: Langfuse 프로젝트 키와 주소(내 PC .env 와 같은 값), 기록의 환경 이름
        "LANGFUSE_PUBLIC_KEY=" + os.environ["LANGFUSE_PUBLIC_KEY"],
        "LANGFUSE_SECRET_KEY=" + os.environ["LANGFUSE_SECRET_KEY"],
        "LANGFUSE_BASE_URL=" + os.environ["LANGFUSE_BASE_URL"],
        "LANGFUSE_TRACING_ENVIRONMENT=production",
    ]
    vm_env_path = output_dir / "vm_serving.env"
    vm_env_path.write_text("\n".join(vm_env_lines) + "\n", encoding="utf-8")

    written_names = []
    for line in vm_env_lines:
        written_names.append(line.split("=")[0])
    return written_names


# 시간 제한은 배포 단원과 같은 90초입니다
written_names = write_vm_serving_env(90)
display(written_names)

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
scp day79_서빙운영_관측_SLO/output/vm_serving.env serving-vm:~/serving-deploy/deploy/.env
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">올린 환경변수는 다음 배포가 컨테이너를 새로 만들 때 적용됩니다.</p>
</div>

### 2-3. 복사, 커밋, push, 배포 확인

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01에서 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/serving-deploy</code> 저장소에 이 단계의 변경 파일만 덮어 복사합니다. 터미널은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료</code> 폴더에서 시작한다고 가정합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pushd</code>로 저장소에 들어가 커밋과 push를 하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">popd</code>로 원래 터미널 위치에 돌아옵니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC 터미널에서 변경을 올립니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cp -R day79_서빙운영_관측_SLO/교안_02_Langfuse_프로덕션_트레이싱/바뀐_파일/serving-deploy/. ~/serving-deploy/
pushd ~/serving-deploy

git add deploy/env.example pyproject.toml serving_app/core/config.py serving_app/routers/metered_router.py serving_app/services/backend.py uv.lock
git commit -m "요청마다 Langfuse trace 남기기"
git push
popd
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/serving-deploy/actions</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">왼쪽 ci의 맨 위 실행(&quot;요청마다 Langfuse trace 남기기&quot;)에서 unit, image 두 상자가 초록 체크가 될 때까지 기다립니다.</li>
<li style="margin:3px 0;">이어서 시작되는 왼쪽 cd의 맨 위 실행에서 deploy 상자가 초록 체크가 될 때까지 기다립니다.</li>
</ol>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">실행이 끝나면 노트북에서 실행 목록을 읽고, ci를 통과해 cd가 배포한 가장 최근 커밋의 sha를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deployed_sha</code>에 담아 둡니다. 6절에서 컨테이너를 직접 다시 띄울 때 이 sha를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code>로 씁니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 실행 중인 이미지의 sha를 저장해 trace의 배포 버전을 확인하고, 뒤의 설정 변경 때 같은 이미지를 다시 띄웁니다.</p>
</div>

In [ ]:
# main push 의 ci 실행과 cd 실행을 나눠 읽습니다. 맨 위가 가장 최근입니다
runs = recent_runs()
ci_runs = runs[(runs["name"] == "ci") & (runs["event"] == "push")]
cd_runs = runs[runs["name"] == "cd"]
display(ci_runs[["display_title", "conclusion", "head_sha", "html_url"]].head(3))
display(cd_runs[["status", "conclusion", "html_url"]].head(3))

# 맨 위 ci(이번 push)가 성공했고, 그 뒤에 생긴 cd(목록에서 그 ci 보다 위)도 성공했는지 확인합니다
assert ci_runs.iloc[0]["conclusion"] == "success", "이번 push 의 ci 가 아직 끝나지 않았거나 실패했습니다. Actions 화면을 보고 1분 뒤 다시 실행하세요."
assert cd_runs.index[0] < ci_runs.index[0] and cd_runs.iloc[0]["conclusion"] == "success", "이번 ci 에 이어진 cd 가 아직 성공하지 않았습니다. 1분 뒤 다시 실행하세요."

# 맨 위 ci 의 커밋이 지금 VM 에서 도는 이미지의 태그입니다(커밋 sha 는 비밀이 아니므로 출력합니다)
deployed_sha = ci_runs.iloc[0]["head_sha"]
print("지금 배포된 sha:", deployed_sha)

## 3. 요청 한 건의 trace

### 3-1. 노무 질문 한 건과 trace 주소

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-app</code>의 키로 노무 질문 하나를 보내고, 응답 헤더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-Trace-Id</code>로 그 trace의 Langfuse 주소를 만듭니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">get_trace_url()</code>은 에이전트 관측 단원에서 쓴 것과 같습니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 요청 한 건의 응답과 Langfuse trace를 ID로 연결해 입력·출력·사용자·지연 기록을 확인합니다.</p>
</div>

In [ ]:
# labor-app 의 키로 노무 질문 하나를 보냅니다
one_response = await user_clients["labor-app"].post("/v1/labor/chat", json={"question": labor_questions[0]["question"]})
one_body = one_response.json()

# 응답 헤더의 trace id 로 Langfuse 화면 주소를 만듭니다
one_trace_id = one_response.headers["X-Trace-Id"]
one_view = {
    "상태 코드": one_response.status_code,
    "답": one_body["answer"],
    "출력 토큰": one_body["usage"]["completion_tokens"],
    "trace 주소": langfuse.get_trace_url(trace_id=one_trace_id),
}
display(one_view)

### 3-2. Langfuse 화면에서 trace 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이번 실습의 User ID에는 발급 표의 숫자 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>가 문자열로 표시됩니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">Langfuse 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://cloud.langfuse.com/project/&lt;프로젝트 ID&gt;/traces/&lt;trace id&gt;</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">왼쪽 트리에서 맨 위 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-chat</code>(span)과 그 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OpenAI-generation</code>을 차례로 누릅니다.</li>
<li style="margin:3px 0;">오른쪽 Preview 탭에서 System(지침과 참고 조문), User(질문), Assistant(모델의 답)를 봅니다.</li>
<li style="margin:3px 0;">위쪽 줄에서 지연(초)과 토큰 수를 보고, 같은 줄의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">+2</code>를 눌러 모델 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code>)을 봅니다.</li>
<li style="margin:3px 0;">맨 위 줄의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user:</code> 값(사용자 키의 번호 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">key_id</code>, 예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">1</code>)과 태그 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor</code>를 봅니다. 배포 버전(Version)은 5-2의 목록 화면에서 열로 봅니다.</li>
</ol>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">화면의 토큰 수는 입력과 출력을 합친 값입니다.</p>
</div>

## 4. 비용 기록

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">Langfuse는 OpenAI 모델처럼 잘 알려진 모델이면 모델 이름으로 단가를 찾아 generation마다 비용을 계산합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code>처럼 직접 서빙하는 모델은 단가가 없어서 3-2의 비용 칸이 비어 있었습니다.</p>
</div>

### 4-1. 출력 토큰 하나의 단가

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU는 시간 단위로 빌리므로, 출력 토큰 하나의 단가는 서빙 벤치마킹 단원의 100만 토큰당 비용 식으로 계산합니다. 이 식은 GPU가 한 시간 내내 쉬지 않고 일한다고 봅니다. 그래서 값은 교안 01 7절 테이블의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">부하 구간만</code> 행과 같습니다. 쉬는 시간까지 넣은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">한 시간에 이 부하 한 번</code> 행보다는 작습니다.</p>
<p style="margin:6px 0;">예를 들어 처리량이 초당 274 토큰이면 GPU는 1시간에 출력 토큰을 274 x 3600개 만듭니다. 시간당 요금을 이 개수로 나눈 값이 출력 토큰 하나의 단가입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/23_cost_definition.png" width="1050" alt="출력 토큰 1개의 단가를 등록한 뒤, 모델 호출 한 건의 출력 토큰 수에 곱해 그 호출의 비용을 계산하는 그림">

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">출력 토큰 하나의 단가</strong> = GPU 시간당 요금 ÷ (출력 처리량 × 3600)<br><strong style="color:#1f3a5f;">generation 하나의 비용</strong> = 출력 토큰 수 × 출력 토큰 하나의 단가</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 처리량:</strong> 교안 01 7절에서 계산한 부하 구간의 값(출력 토큰 / 구간 초)을 아래 셀에 숫자로 적습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력 단가는 0:</strong> 이 교안은 GPU 시간 요금을 모두 출력 토큰에 나눠 매깁니다. 같은 GPU 시간이 입력 처리와 출력 생성에 함께 쓰여 둘로 나눌 기준이 없기 때문입니다. 그래서 비용 합은 출력 토큰 합에 비례합니다.</li>
</ul>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> GPU 시간당 요금과 앞 교안에서 얻은 출력 처리량으로 토큰 단가를 계산해, generation별 비용을 Langfuse에 붙입니다.</p>
</div>

In [ ]:
# 시간당 요금(달러). 교안 01 1-1에서 .env 에 적은 Total / hour 값입니다
GPU_PRICE_PER_HOUR = float(os.environ["GPU_PRICE_PER_HOUR"])

MEASURED_OUTPUT_TOKENS_PER_SECOND = 274  # 예: 서빙 벤치마킹 단원 L4 측정(동시 16, 캐싱 켬). 교안 01 7절에서 계산한 내 값으로 바꿉니다

# 출력 토큰 하나의 단가 = 시간당 요금 / (출력 처리량 * 3600)
output_price_per_token = GPU_PRICE_PER_HOUR / (MEASURED_OUTPUT_TOKENS_PER_SECOND * 3600)

price_view = {
    "출력 토큰 하나의 단가(달러)": output_price_per_token,
    "100만 출력 토큰당(달러)": round(output_price_per_token * 1_000_000, 3),
}
display(price_view)

### 4-2. Langfuse에 모델 단가 등록

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델 정의(model definition)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">Langfuse 프로젝트에 모델 이름 규칙(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">match_pattern</code>, 정규식)과 사용량 종류(input, output)별 단가를 등록한 것입니다. generation의 모델 이름이 규칙에 맞으면 Langfuse가 그 단가로 비용을 계산합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">소급 없음:</strong> 단가는 등록한 뒤 들어온 기록에만 적용되고, 이미 들어온 기록의 비용은 빈 채로 남습니다. 서버가 비용을 직접 넣어 보낸 기록은 그 값이 단가 계산보다 우선합니다(이 서버는 넣지 않습니다).</li>
</ul>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 직접 서빙한 모델의 출력 토큰 단가를 등록해 이후 generation 기록에 비용이 계산되도록 합니다.</p>
</div>

In [ ]:
from langfuse.api import PricingTierInput

# 모델 이름이 qwen3.5-4b 인 generation 에 출력 토큰 단가를 붙입니다
model_price = langfuse.api.models.create(
    model_name=SERVED_NAME,
    match_pattern=r"(?i)^qwen3\.5-4b$",   # 대소문자 무시, 이름 전체가 같을 때만
    unit="TOKENS",
    pricing_tiers=[
        # is_default=True: 조건(conditions) 없이 늘 쓰는 기본 단가. prices 의 키는 Usage 의 종류(input, output)
        PricingTierInput(name="기본", is_default=True, priority=0, conditions=[], prices={"input": 0.0, "output": output_price_per_token}),
    ],
)
display({"id": model_price.id, "model_name": model_price.model_name, "match_pattern": model_price.match_pattern})

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">이 셀은 한 번만 실행합니다. 같은 이름의 모델 정의는 하나만 둘 수 있어, 다시 실행하면 새 정의가 생기지 않고 400 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Model name 'qwen3.5-4b' already exists in project</code>로 거절됩니다. 이 400이 보이면 단가가 이미 등록된 것이므로 그대로 다음 셀로 넘어갑니다. 단가를 바꿔야 할 때만 Settings &gt; Model Definitions에서 그 행을 지운 뒤 다시 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">3-1의 기록:</strong> 단가를 등록하기 전에 들어왔으므로 비용이 빈 채로 남습니다. 5절 부하부터 비용이 붙습니다.</li>
</ul>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">Langfuse 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://cloud.langfuse.com/project/&lt;프로젝트 ID&gt;/settings/models</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">Langfuse에 기본으로 들어 있는 모델 목록 위에 방금 등록한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> 행(Maintained by가 User)이 있는지 봅니다.</li>
<li style="margin:3px 0;">Prices per unit 칸이 좁아 output 단가가 잘려 보이므로 행을 눌러 상세 화면을 열고, Pricing의 output 단가가 위 셀의 단가와 같은지 봅니다.</li>
</ol>
</div>

## 5. trace를 데이터로 읽기

### 5-1. 노무 부하 64건

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01 4절과 같은 방법으로 노무 질문 64건을 동시 16으로 보냅니다. 질문을 pro 키 셋에 번갈아 나눠, 키 하나가 60초에 30번 한도를 넘지 않게 합니다. 5-2에서 이 부하의 기록만 읽을 수 있도록 시작과 끝 시각을 UTC로 남깁니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 요청 결과와 UTC 시작·끝 시각을 기록해 이번 부하의 trace만 모으고, 뒤에서 지연과 사용자별 사용량을 계산합니다.</p>
</div>

In [ ]:
# 부하 시작 시각(UTC). Langfuse 는 시각을 UTC 로 저장합니다
load_start = datetime.now(timezone.utc)

# 노무 질문 64개를 pro 키 셋에 번갈아 나눠 동시에 보냅니다
load_tasks = []
for index, item in enumerate(labor_questions[:64]):
    client = labor_clients[index % len(labor_clients)]
    load_tasks.append(run_limited(send_chat(client, "/v1/labor/chat", {"question": item["question"]})))
load_rows = pd.DataFrame(await asyncio.gather(*load_tasks))

load_end = datetime.now(timezone.utc)

# 상태 코드별 요청 수
display(load_rows["상태 코드"].value_counts())

### 5-2. 이번 부하의 기록 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">observation 목록 API</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">langfuse.api.observations.get_many()</code>는 프로젝트의 observation을 조건(종류, 시작 시각 범위)으로 걸러 돌려줍니다. 아래 함수는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">limit=100</code>으로 한 번에 100개씩 받고, 더 있으면 응답의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">meta.cursor</code>(다음 쪽의 시작 위치)를 다음 요청에 넘깁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">fetch_generations(start, end)</code>:</strong> 준비 셀에서 정의한 함수로, 이 API를 쪽마다 불러 generation 한 개를 한 행(trace id, user_id, version, tags, level, 지연, 출력 토큰, 비용)으로 모읍니다. 아래에서 이번 부하의 p95와 사용자별 합계를 계산할 때 씁니다.</li>
</ul>
<p style="margin:6px 0;">서버는 기록을 모았다가 보내므로 부하가 끝난 직후에는 일부만 들어와 있을 수 있습니다. 아래 셀은 30초마다 기록을 읽고, 출력 토큰이 있는 행(성공한 모델 호출) 수가 성공 응답 수 이상이 되면 멈춥니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 성공한 모델 호출의 trace가 모두 도착했는지 확인해, 불완전한 자료로 지연이나 사용자별 합계를 계산하지 않게 합니다.</p>
</div>

In [ ]:
# 성공 응답 수만큼 기록이 들어올 때까지 30초마다 읽습니다(최대 10번, 약 5분)
success_count = int((load_rows["상태 코드"] == 200).sum())
for attempt in range(10):
    generations = fetch_generations(load_start, load_end)
    print(attempt + 1, "번째 읽기:", generations["출력 토큰"].notna().sum(), "행 / 성공 응답", success_count, "건")
    if generations["출력 토큰"].notna().sum() >= success_count:
        break
    time.sleep(30)

# 최대 10회 읽은 뒤에도 성공 응답 수만큼 기록이 없으면, 불완전한 표로 계산하지 않습니다
received_count = int(generations["출력 토큰"].notna().sum())
if received_count < success_count:
    raise RuntimeError(f"Langfuse 기록이 아직 덜 도착했습니다: 성공 응답 {success_count}건 중 {received_count}건만 확인했습니다. 잠시 뒤 이 셀을 다시 실행하세요.")

display(generations.head())

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">Langfuse 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://cloud.langfuse.com/project/&lt;프로젝트 ID&gt;</code>(프로젝트 화면)</li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">목록의 행마다 Latency, Cost, Version(2-3의 sha), User ID(키 번호)가 채워져 있는지 봅니다. 위 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">generations</code>에 든 기록이 이 목록의 맨 위쪽에 있습니다.</li>
<li style="margin:3px 0;">맨 위 행을 눌러 trace 상세를 열고, 왼쪽 트리의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OpenAI-generation</code> 줄에 지연과 비용이 붙었는지 봅니다. 3-2의 기록과 달리 4-2에서 등록한 단가로 비용이 계산되어 있습니다.</li>
</ol>
</div>

### 5-3. 정확한 p95

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">히스토그램에는 개별 요청의 지연이 남지 않아 p95를 정확한 값으로 알 수 없습니다. 기록에는 요청마다 지연 값이 있으므로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quantile(0.95)</code>로 p95를 바로 계산합니다. 95% 위치가 두 요청 사이에 걸리면 두 값 사이를 비례로 나눕니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">generation의 지연:</strong> 서빙 API가 vLLM을 부르고 답을 받기까지의 시간입니다. 키 확인과 사용량 기록 시간은 빠져 있어 서빙 API 응답 시간(교안 01 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_request_seconds</code>)보다 조금 짧습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 걸린 시간:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">send_chat</code>이 측정한 값으로, 인터넷 왕복과 VM 처리 시간까지 들어갑니다.</li>
</ul>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> Langfuse와 내 PC의 평균·p95를 비교해 모델 호출 시간과 네트워크·서빙을 포함한 전체 지연을 구분합니다.</p>
</div>

In [ ]:
# 두 자료의 평균, p95, 15초 이하 비율을 같은 계산으로 나란히 봅니다
latency_rows = []
for source, seconds in {"Langfuse generation 지연": generations["지연(초)"], "내 PC 걸린 시간": load_rows["걸린 시간(초)"]}.items():
    latency_rows.append({
        "자료": source,
        "요청 수": len(seconds),
        "평균(초)": round(seconds.mean(), 2),
        "p95(초)": round(seconds.quantile(0.95), 2),
        "15초 이하 비율": round((seconds <= 15).mean(), 3),
    })
latency_table = pd.DataFrame(latency_rows)
display(latency_table)

### 5-4. 사용자별 토큰과 비용 합계

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이번 부하의 기록을 사용자별로 묶어, 누가 요청을 얼마나 보내고 토큰과 비용을 얼마나 썼는지 비교합니다. 이미 읽은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">generations</code>를 그대로 사용하므로 단건 시연이나 다른 시각의 요청은 합계에 섞이지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 사용자별 요청·출력 토큰·비용을 집계해 사용량과 비용이 어느 키에 귀속되는지 확인합니다.</p>
</div>

In [ ]:
# 이번 부하의 사용자(key_id)별 요청 수, 출력 토큰 합, 비용 합
user_table = generations.groupby("user_id").agg(
    requests=("trace_id", "count"),
    output_tokens=("출력 토큰", "sum"),
    cost=("비용(달러)", "sum"),
)

# 발급 응답에 있는 key_id 로 사용자 이름을 붙입니다
key_user_names = {}
for user, issued in issued_keys.items():
    key_id = str(issued["key_id"])  # 키 번호(정수)를 기록의 user_id 와 같은 글자로 맞춥니다
    key_user_names[key_id] = user
user_table["user"] = pd.Series(key_user_names)

display(user_table.round(6))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 단가를 쓰므로 출력 토큰을 많이 쓴 사용자의 비용도 큽니다. 실패한 모델 호출은 출력 토큰이 비어 있어 토큰·비용 합에 더해지지 않습니다.</p>
</div>

## 6. 실패한 요청

### 6-1. 시간 제한 1초로 504 만들기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">FastAPI 서빙 API 단원에서 vLLM이 시간 제한 안에 답하지 않으면 504(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">backend_timeout</code>)가 오는 것을 보았습니다. VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>만 1초로 바꿔 컨테이너를 다시 띄우면, 거의 모든 상담이 1초 안에 끝나지 않아 504가 됩니다. 이 실패가 Langfuse에 어떻게 남는지 봅니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 일부러 만든 504 요청의 시각과 Langfuse ERROR 기록을 대조해 실패한 모델 호출이 어떻게 저장되는지 확인합니다.</p>
</div>

In [ ]:
# 시간 제한만 1초로 바꾼 .env 를 씁니다
written_names = write_vm_serving_env(1)
display(written_names)

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
scp day79_서빙운영_관측_SLO/output/vm_serving.env serving-vm:~/serving-deploy/deploy/.env
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# VM 에서 지금 배포된 sha 로 컨테이너를 다시 띄우는 명령(IMAGE_TAG 를 빼면 낡은 latest 가 뜹니다)
print("IMAGE_TAG=" + deployed_sha + " docker compose up -d")

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~/serving-deploy/deploy

# 위 셀이 출력한 한 줄을 그대로 붙여 넣습니다. .env 가 바뀌어 compose 가 컨테이너를 새로 만듭니다
IMAGE_TAG=<위 셀이 출력한 sha> docker compose up -d
docker compose ps
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose ps</code>의 STATUS가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Up</code>이면 노무 질문 세 건을 보냅니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 시간 제한을 1초로 둔 동안의 응답 상태와 시각을 기록해, 해당 실패의 Langfuse generation을 찾습니다.</p>
</div>

In [ ]:
# 실패 구간의 시작 시각을 남기고 노무 질문 세 건을 보냅니다
fail_start = datetime.now(timezone.utc)
fail_tasks = []
for item in labor_questions[:3]:
    fail_tasks.append(run_limited(send_chat(user_clients["mobile-app"], "/v1/labor/chat", {"question": item["question"]})))
fail_rows = pd.DataFrame(await asyncio.gather(*fail_tasks))
fail_end = datetime.now(timezone.utc)
display(fail_rows)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">504 / backend_timeout</code>은 서빙 API가 1초 안에 답을 받지 못했다는 뜻입니다. 실패 응답에는 trace id 헤더가 없으므로 요청을 보낸 시각으로 기록을 찾습니다.</p>
</div>

### 6-2. 오류 레벨로 남은 기록

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">레벨(level)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">observation마다 붙는 중요도 표시이고, DEBUG, DEFAULT, WARNING, ERROR 네 가지가 있습니다. 정상 기록은 DEFAULT입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">langfuse.openai</code>는 모델 호출이 예외로 끝나면 그 generation의 레벨을 ERROR로 두고, 예외 메시지를 상태 메시지(status message)에 남깁니다.</li>
<li style="margin:3px 0;">예를 들어 6-1에서 504로 끝난 요청은 1초 안에 답을 받지 못해 시간 초과 예외로 끝났으므로, 그 generation이 ERROR로 남습니다.</li>
</ul>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 실패 구간의 generation 중 ERROR만 골라 시간 초과 기록의 레벨과 상태 메시지를 확인합니다.</p>
</div>

In [ ]:
# 실패 구간의 기록을 읽고 레벨이 ERROR 인 행만 봅니다(기록이 들어오도록 30초 기다립니다)
time.sleep(30)
fail_generations = fetch_generations(fail_start, fail_end)
error_generations = fail_generations[fail_generations["level"] == "ERROR"]
display(error_generations)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">시간 초과로 끝난 모델 호출은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ERROR</code>로 기록됩니다. 답을 받지 못해 출력 토큰 사용량은 비어 있습니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">Langfuse 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://cloud.langfuse.com/project/&lt;프로젝트 ID&gt;</code>(프로젝트 화면)</li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">Filter에서 Level을 ERROR로 고르면 오류 레벨 observation이 든 trace만 남습니다.</li>
<li style="margin:3px 0;">generation을 눌러 Error 표시와 상태 메시지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Request timed out.</code>을 봅니다.</li>
</ol>
</div>

### 6-3. 시간 제한 되돌리기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">반드시 원래 값 90초로 되돌려 다시 띄웁니다.</strong> 1초로 두면 이 뒤의 모든 상담이 504로 끝납니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 시간 제한을 원래 값으로 되돌린 .env 를 씁니다
written_names = write_vm_serving_env(90)
display(written_names)

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
scp day79_서빙운영_관측_SLO/output/vm_serving.env serving-vm:~/serving-deploy/deploy/.env
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~/serving-deploy/deploy

# 6-1 에서 붙여 넣은 줄을 다시 붙여 넣습니다
IMAGE_TAG=<6-1 의 sha> docker compose up -d
docker compose ps
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 원래 시간 제한을 복구한 뒤 정상 응답을 확인해, 다음 부하 실습을 계속할 수 있는 상태인지 점검합니다.</p>
</div>

In [ ]:
# 되돌린 뒤 노무 질문 한 건이 200 인지 확인합니다
restore_row = await send_chat(user_clients["mobile-app"], "/v1/labor/chat", {"question": labor_questions[3]["question"]})
display(restore_row)

## 7. 🖐️ 함께 따라하기: 전월세 창구의 사용자별 토큰과 비용

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습에서 노무 창구로 한 부하 보내기, 기록 읽기, 사용자별 합계를 전월세 창구로 직접 작성합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">주소:</strong> 전월세 상담 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/lease/chat</code>입니다. 기록의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">['lease']</code>이고 span 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-chat</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/따라하기/lease_questions.jsonl</code>의 앞 16건입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">고르는 방법:</strong> 시각 범위로 읽은 기록에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease</code>가 든 행만 고릅니다. 두 창구를 한 시각 범위에서 함께 읽을 때 창구를 나누는 방법입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 작성 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>만 주석에 맞춰 채웁니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_02_정답.ipynb</code>의 같은 절에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정:</strong> 이 절에서 전월세 사용자 키를 따로 발급합니다. 동시 요청 수는 본실습과 같습니다.</p>
</div>

### 7-1. 전월세 부하 16건

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">전월세 사용자 두 명의 pro 키를 발급하고, 각 키를 담은 클라이언트를 만듭니다. 키 원문은 표에 넣지 않습니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 전월세 사용자를 분리해 발급하고, 뒤의 부하 요청과 사용자별 집계에 사용할 클라이언트를 준비합니다.</p>
</div>

In [ ]:
lease_users = ["lease-app", "lease-web"]
lease_keys = {}
lease_key_rows = []
for user in lease_users:
    response = httpx.post(VM_URL + "/v1/admin/keys", headers=ADMIN_HEADERS, json={"user": user, "plan": "pro"}, timeout=REQUEST_TIMEOUT)
    issued = response.json()
    lease_keys[user] = issued["api_key"]
    lease_key_rows.append({"상태 코드": response.status_code, "key_id": issued["key_id"], "사용자": issued["user"], "요금제": issued["plan"]})

lease_clients = []
for user in lease_users:
    lease_clients.append(httpx.AsyncClient(base_url=VM_URL, headers={"X-API-Key": lease_keys[user]}, timeout=REQUEST_TIMEOUT))
display(pd.DataFrame(lease_key_rows))

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 5-1의 부하 보내기 셀을 참고합니다.
질문마다 `lease_clients`의 키를 번갈아 쓰고, 전월세 상담 주소를 `send_chat`에 전달합니다.
```

</details>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 전월세 부하의 상태 코드와 UTC 시간 범위를 기록해, 해당 요청의 trace를 뒤에서 골라냅니다.</p>
</div>

In [ ]:
# 전월세 질문을 읽습니다
lease_questions = load_jsonl(data_dir / "따라하기" / "lease_questions.jsonl")

# 함께 따라하기: 전월세 상담 주소로 16건을 pro 키 셋에 나눠 보내세요
lease_start = datetime.now(timezone.utc)
lease_tasks = []
for index, item in enumerate(lease_questions[:16]):
    client = lease_clients[index % len(lease_clients)]
    lease_tasks.append(run_limited(send_chat(client, ..., {"question": item["question"]})))
lease_rows = pd.DataFrame(await asyncio.gather(*lease_tasks))
lease_end = datetime.now(timezone.utc)
display(lease_rows["상태 코드"].value_counts())

### 7-2. 전월세 기록만 골라 사용자별 합계

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 5-2와 5-4를 참고합니다.
세부구현: tags 열의 값은 리스트이거나 None 입니다. None 이 아닌지 먼저 보고, 리스트에 "lease"가 들어 있는지를 in 으로 확인해 True, False 목록을 만듭니다. 합계는 5-4처럼 user_id 로 묶어 trace_id 개수, 출력 토큰 합, 비용 합을 구합니다.
```

</details>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 전월세 trace만 골라 사용자별 요청 수·출력 토큰·비용을 계산합니다.</p>
</div>

In [ ]:
# 성공 응답 수만큼 기록이 들어올 때까지 30초마다 읽습니다(최대 10번, 5-2 와 같은 모양)
lease_success = int((lease_rows["상태 코드"] == 200).sum())
for attempt in range(10):
    lease_generations_all = fetch_generations(lease_start, lease_end)
    print(attempt + 1, "번째 읽기:", lease_generations_all["출력 토큰"].notna().sum(), "행 / 성공 응답", lease_success, "건")
    if lease_generations_all["출력 토큰"].notna().sum() >= lease_success:
        break
    time.sleep(30)

# 함께 따라하기: tags 가 None 이 아니고 "lease" 가 든 행이면 True 인 목록을 만드세요
is_lease = []
for tags in lease_generations_all["tags"]:
    is_lease.append(...)
lease_generations = lease_generations_all[is_lease]

# 함께 따라하기: 사용자(key_id)별 요청 수, 출력 토큰 합, 비용 합을 구하세요
lease_user_table = ...
display(lease_user_table.round(6))

## 8. 핵심 정리

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 안의 trace:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">from langfuse.openai import AsyncOpenAI</code> 한 줄로 모델 호출마다 generation이 남고, 상담 요청을 span(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-chat</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-chat</code>)으로 감싸 한 trace로 묶습니다. Langfuse 키가 없는 CI에서는 기록 없이 테스트가 통과합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">propagate_attributes</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user_id</code>(key_id, US-ASCII 200자 이하), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>(창구), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>(이미지 태그 = 커밋 sha), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">metadata</code>(plan, think)를 span과 generation에 함께 붙입니다. 환경은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">LANGFUSE_TRACING_ENVIRONMENT=production</code>입니다. 응답 헤더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-Trace-Id</code>로 trace id를 돌려줍니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">VM .env 아홉 줄:</strong> 배포 단원의 다섯 줄에 Langfuse 네 줄을 더합니다. push 전에 scp로 올려 두면 배포 작업이 새 컨테이너를 만들 때 읽습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비용:</strong> 출력 토큰 하나의 단가 = GPU 시간당 요금 / (출력 처리량 x 3600)을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">models.create</code>에 등록합니다(GPU가 쉬지 않는다고 본 단가, 입력 단가 0). 등록 뒤 들어온 기록부터 비용이 붙습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">API로 읽기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">fetch_generations(start, end)</code>로 이번 부하의 기록을 테이블로 받아 정확한 p95와 사용자별 토큰, 비용 합을 계산합니다. 개별 기록으로 사용자별 사용량과 느린 요청의 원인을 분석합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실패한 요청:</strong> 시간 초과로 끝난 모델 호출은 레벨 ERROR와 상태 메시지로 남습니다. 실패 응답에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-Trace-Id</code>가 없으므로 시각 범위와 Level 필터로 찾습니다.</li>
</ul>
<p style="margin:6px 0;">VM의 시간 제한이 90초로 돌아왔는지 확인하고, GPU Pod와 VM은 켜 둡니다.</p>
</div>